In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence

from sklearn.metrics import roc_auc_score
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sksurv.metrics import concordance_index_censored
from sksurv.util import Surv

from ncps.torch import LTC
from ncps.wirings import AutoNCP

RAND_SEED = 5904
torch.manual_seed(RAND_SEED)
torch.cuda.manual_seed(RAND_SEED)

## Load Data

In [ ]:
split_data = np.load("../data/train_test_splits.npz", allow_pickle=True)

train_X = split_data["train_X"]
train_Y = split_data["train_Y"]
test_X = split_data["test_X"]
test_Y = split_data["test_Y"]

NUM_FEATURES = train_X.shape[1]

## LTC LNN Model

In [ ]:
def init_weights(module):
    if isinstance(module, nn.Linear):
        torch.nn.init.xavier_normal_(module.weight)

        if module.bias is not None:
            nn.init.constant_(module.bias, 0)

class ProbabilityPredictorLTC(nn.Module):
    def __init__(self, input_size, num_neurons, wiring : AutoNCP, batch_first=True, return_sequences=True, ode_unfolds=6):
        super(ProbabilityPredictorLTC, self).__init__()

        self.num_neurons = num_neurons

        self.ltc_lnn = LTC(input_size=input_size,
                            units=wiring,
                            batch_first=batch_first,
                            return_sequences=return_sequences,
                            ode_unfolds=ode_unfolds)
        
        self.fc1 = nn.Linear(wiring.output_dim, 1)
        # self.relu = nn.ReLU()
        # self.tanh = nn.Tanh()

        # self.dropout1 = nn.Dropout(0.2)
        self.dropout2 = nn.Dropout(0.3)

        self.sigmoid = nn.Sigmoid()
    
        self.apply(init_weights)

    def forward(self, input, timespans):

        x, _ = self.ltc_lnn(input=input, hx=None, timespans=timespans)
        # x = self.tanh(x)
        # x = self.dropout1(x)
        x = self.fc1(x)
        x = self.dropout2(x)
        x = self.sigmoid(x)

        return x

def get_ltc_model(num_inputs, num_outputs, num_neurons, network_sparsity=0.5, ode_unfolds=6, return_sequences=True):
    
    network_wiring = AutoNCP(num_neurons, num_outputs, sparsity_level=network_sparsity, seed=RAND_SEED)

    model = ProbabilityPredictorLTC(input_size=num_inputs,
                                    num_neurons=num_neurons,
                                    wiring=network_wiring,
                                    batch_first=True,
                                    return_sequences=return_sequences,
                                    ode_unfolds=ode_unfolds)
    
    return model

In [ ]:
def get_data_sequences(features, event_times, labels, num_neurons, t_step):
    feature_sequences = list()
    time_sequences = list()
    label_sequences = list()
    sequence_masks = list()

    for feature_vector, time, label in zip(features, event_times, labels):

        divisions = time / t_step

        whole_divisions = int(divisions)
        remainder_divisions = divisions % 1

        num_time_steps = whole_divisions

        if whole_divisions > 0:
            time_seq = np.stack([t_step] * num_time_steps)
        else:
            time_seq = np.array([])

        if remainder_divisions > 0:
            time_seq = np.append(time_seq, (remainder_divisions * t_step))
            num_time_steps += 1

        assert time_seq.sum() == time

        time_seq = torch.tensor(time_seq)
        time_sequences.append(time_seq)

        feature_seq = np.stack([feature_vector] * num_time_steps)
        feature_seq = torch.tensor(feature_seq)
        feature_sequences.append(feature_seq)

        label_seq = np.zeros_like(time_seq)
        label_seq[-1] = label
        label_seq = torch.tensor(label_seq)
        label_sequences.append(label_seq)

        seq_mask = np.ones_like(time_seq)
        seq_mask = torch.tensor(seq_mask)
        sequence_masks.append(seq_mask)

    times_T = pad_sequence(time_sequences, batch_first=True, padding_value=1e-8, padding_side="right")
    times_T = np.expand_dims(times_T, axis=-1)
    times_T = np.broadcast_to(times_T, (times_T.shape[0], times_T.shape[1], num_neurons))
    times_T = torch.tensor(times_T, dtype=torch.float32)

    features_X = pad_sequence(feature_sequences, batch_first=True, padding_value=0, padding_side="right")
    features_X = features_X.type(torch.float32)

    labels_Y = pad_sequence(label_sequences, batch_first=True, padding_value=0, padding_side="right")
    labels_Y = np.expand_dims(labels_Y, axis=-1)
    labels_Y = torch.tensor(labels_Y, dtype=torch.float32)

    masks_M = pad_sequence(sequence_masks, batch_first=True, padding_value=0, padding_side="right")
    masks_M = np.expand_dims(masks_M, axis=-1)
    masks_M = torch.tensor(masks_M, dtype=torch.float32)

    return features_X, times_T, labels_Y, masks_M

In [ ]:
# class RelapseDataset(torch.utils.data.Dataset):
#     def __init__(self, X, dt, Y):
#         self.featuresX = torch.tensor(X, dtype=torch.float32)
#         self.time = torch.tensor(dt, dtype=torch.float32)
#         self.relapseOutcome = torch.tensor(Y, dtype=torch.float32)

#     def __len__(self):
#         return len(self.featuresX)

#     def __getitem__(self, idx):
#         return self.featuresX[idx], self.time[idx], self.relapseOutcome[idx]

def get_ltc_dataset(num_neurons, time_step):
    # Stacking features along 2nd dimension to match the number of time steps

    train_features_X, train_times_T, train_labels_Y, train_masks_M = get_data_sequences(train_X, train_Y[:, 1], train_Y[:, 0], num_neurons, time_step)
    test_features_X, test_times_T, test_labels_Y, test_masks_M = get_data_sequences(test_X, test_Y[:, 1], test_Y[:, 0], num_neurons, time_step)

    # print(test_features_X.shape)
    # print(test_times_T.shape)
    # print(test_labels_Y.shape)

    training_data = torch.utils.data.TensorDataset(train_features_X, train_times_T, train_labels_Y, train_masks_M)
    testing_data = torch.utils.data.TensorDataset(test_features_X, test_times_T, test_labels_Y, test_masks_M)

    trainloader = torch.utils.data.DataLoader(training_data, batch_size=16, shuffle=True)
    testloader = torch.utils.data.DataLoader(testing_data, batch_size=16, shuffle=False)

    return trainloader, testloader

In [ ]:
def get_perf_metrics(model, dataloader, device=torch.device("cpu")):
    predictions_y_hat = []
    true_y = []
    time_vals = []

    with torch.no_grad():
        model.eval()
        model.ltc_lnn.return_sequences = True
        for data in dataloader:
            features, times, labels, masks = data

            features = features.to(device)
            times = times.to(device)
            labels = labels.to(device)
            masks = masks.to(device)

            output = model(input=features, timespans=times)
            output = output.squeeze(-1)

            batch_size = output.shape[0]
            batch_indexes = torch.arange(0, batch_size)


            event_indexes = masks.sum(dim=1) - 1
            event_indexes = event_indexes.reshape(-1).tolist()
            
            preds_at_event = output[batch_indexes, event_indexes].tolist()

            truths = labels.sum(dim=1).reshape(-1).tolist()
            event_times_t = times.sum(dim=1).T[0].tolist()

            # # Append to predictions list
            predictions_y_hat = predictions_y_hat + preds_at_event
            true_y = true_y + truths
            time_vals = time_vals + event_times_t

    # roc = roc_auc_score(true_y, predictions_y_hat)
    # mse = mean_squared_error(true_y, predictions_y_hat)
    rmse = root_mean_squared_error(true_y, predictions_y_hat)
    concordance_data = concordance_index_censored(np.array(true_y).astype(bool), time_vals, predictions_y_hat)

    # print("ROC: {}".format(roc))
    # # print("MSE: {}".format(mse))
    print("RMSE: {}".format(rmse))
    print("C-Index: {}".format(concordance_data[0]))

    model.train()

## Training Model

In [ ]:
BATCH_SIZE = 64
BATCH_PRINT_STEP = 10
NUM_EPOCHS = 100

if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)
print("Training Over", len(train_X), "follow ups")

num_neurons = 64
num_outputs = 16

model = get_ltc_model(NUM_FEATURES, num_outputs, num_neurons, network_sparsity=0.5, return_sequences=True)
trainloader, testloader = get_ltc_dataset(num_neurons, time_step=1)

model = model.to(device)

criterion = nn.BCELoss(reduction="none")
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001)

for epoch in range(0, NUM_EPOCHS):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):
        features, times, labels, masks = data

        features = features.to(device)
        times = times.to(device)
        labels = labels.to(device)
        masks = masks.to(device)
        
        # Zero gradients
        optimizer.zero_grad()

        # Forward
        output = model(input=features, timespans=times)

        # Backward
        raw_loss = criterion(output, labels)

        masked_loss = masks * raw_loss
        loss = masked_loss.sum() / masks.sum()
        loss.backward()
        
        # Optimize
        optimizer.step()

        running_loss += loss.item()

        if i % BATCH_PRINT_STEP == (BATCH_PRINT_STEP - 1):
            curr_loss = running_loss / BATCH_PRINT_STEP
            epochBatchLossPrint = "Epoch: {} Batch: {} Loss: {:.5f}".format(epoch + 1, i + 1, curr_loss)
            print(epochBatchLossPrint)
            running_loss = 0.0
    

    print("Training Data:")
    get_perf_metrics(model=model, dataloader=trainloader, device=device)
    print("Testing Data:")
    get_perf_metrics(model=model, dataloader=testloader, device=device)
    

In [ ]:
if(torch.cuda.is_available()):
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Device:", device)

model = model.to(device)

print("Training Data:")
get_perf_metrics(model=model, dataloader=trainloader, device=device)
print("Testing Data:")
get_perf_metrics(model=model, dataloader=testloader, device=device)

In [ ]:
torch.save(model.state_dict(), "weights_LTC.pth")

In [ ]:
months = 24
num_steps = 24

time_diff = float(months) / float(num_steps)

# print(train_features_X[0].shape)
# print(train_times_T[0].shape)

# Convert a single test case to the right shape [Batch, Time, Neuron]
# In the ode solver, the time is used for calculations in each neuron.
# The calculations are performed elementwise so 
# Create list of time points

timespans_list = [time_diff] * num_steps

# Convert to numpy array
single_test_t = np.stack(timespans_list)

single_test_t = np.expand_dims(single_test_t, axis=-1)
single_test_t = np.broadcast_to(single_test_t, (single_test_t.shape[0], num_neurons))
single_test_t = torch.tensor(single_test_t).float().unsqueeze(0)
# print(single_test_t.shape)

# print(test_Y[:,0][test_index])

# Convert a single test case to the right shape [Batch, Vector, Feature]
single_feature_vector = test_X[10]

# Make copies of the feature vector to match the number of prediction time points
copied_vectors = [single_feature_vector] * len(timespans_list)

single_test_X = np.stack(copied_vectors, axis=0)
# print(single_test_X.shape)
single_test_X = torch.tensor(single_test_X).float().unsqueeze(0)
# print(single_test_X)

with torch.no_grad():
    model.eval()
    model.to("cpu")

    model.ltc_lnn.return_sequences = True
    pred = model(input=single_test_X, timespans=single_test_t)
    model.ltc_lnn.return_sequences = False
    
pred_flatten = pred.reshape(-1).numpy()

print(pred_flatten.shape)
print(pred_flatten)

In [ ]:
x_values = np.arange(time_diff, months+time_diff, time_diff)

pred_rescaled = np.multiply(pred_flatten, time_diff)
cumulative_hazard = np.cumsum(pred_rescaled)

surv_func = np.exp(-cumulative_hazard)

plt.figure()
plt.step(x_values, surv_func, where="post")
plt.xlim([0, months])
plt.ylim([0, 1])
# plt.plot(x_values, surv_func)
plt.show()